# STICS v11 Successive Converter - Test Run

This notebook tests rotations and long-term experiments where one `idsim` references several `CropManagement` seasons ordered by `SeasonOrder`.

## 1. Import Required Modules

In [1]:
import os
import sys
from pathlib import Path
import sqlite3
from modfilegen import GlobalVariables
from modfilegen.Converter.SticsV11Converter.sticsconverter import main
from modfilegen.Converter.SticsV11Converter.sticssuccessiveconverter import fetch_rotation_seasons

## 2. Configure Paths and Global Variables

Set the required paths for:
- `dbMasterInput`: Path to the master input SQLite database
- `dbModelsDictionary`: Path to the models dictionary database
- `directorypath`: Output directory for generated STICS files
- `pltfolder`: Path to cultivar/plant parameter files
- Other configuration parameters

In [2]:
# Base paths derived from the repository root, not from the kernel cwd
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    (
        path
        for path in candidates
        if (path / "tests" / "successive").exists() and (path / "src" / "modfilegen").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        f"Could not locate the ModFileGen repository from cwd={cwd}."
    )

data_dir = repo_root / "tests" / "successive"
output_dir = data_dir / "output"
temp_dir = output_dir / "temp"
package_dir = repo_root / "src" / "modfilegen"

# Create output directories if they don't exist
output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(parents=True, exist_ok=True)
print(f"Kernel cwd: {cwd}")
print(f"Repo root: {repo_root}")
print(f"Output directory: {output_dir.resolve()}")

# Database paths
master_input_db = data_dir / "MasterInput.db"
models_dict_db = data_dir /  "ModelsDictionaryArise.db"

# Cultivar/plant parameters folder
cultivars_folder = data_dir / "cultivars" / "stics"

# Configuration
n_threads = 2  # Number of parallel workers
n_parts = 1    # Sub-batches per worker
dt = 0        # choice to delete intermediate files 0 (no) or 1 (yes)
dailyoutput = 1  # Store daily and profile STICS outputs

# Verify files exist
print("Checking files...")
print(f"Master Input DB exists: {master_input_db.exists()}")
print(f"Models Dict DB exists: {models_dict_db.exists()}")
print(f"Cultivars folder exists: {cultivars_folder.exists()}")

Kernel cwd: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/notebooks
Repo root: /mnt/d/Mes Donnees/TCMP/github/ModFileGen
Output directory: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/output
Checking files...
Master Input DB exists: True
Models Dict DB exists: True
Cultivars folder exists: True


## 3. Set Global Variables

Configure the GlobalVariables dictionary that will be used by the converter.

In [3]:
# Run only the three-season rotation used by this notebook test
target_idsim = "5.925_6.025_2000_ROT_MAIZE_PEANUT_3Y_2"

# Set global variables
GlobalVariables["dbMasterInput"] = str(master_input_db)
GlobalVariables["dbModelsDictionary"] = str(models_dict_db)
GlobalVariables["directorypath"] = str(output_dir)
GlobalVariables["pltfolder"] = str(cultivars_folder)
GlobalVariables["nthreads"] = n_threads
GlobalVariables["parts"] = n_parts
GlobalVariables["dt"] = dt
GlobalVariables["dailyoutput"] = dailyoutput
GlobalVariables["tempDir"] = str(temp_dir)
GlobalVariables["package"] = str(package_dir)
GlobalVariables.pop("sticsIdsim", None)  # Run all SimUnitList rows
GlobalVariables.pop("skipSecondRun", None)

print("Global Variables configured:")
for key, value in GlobalVariables.items():
    print(f"  {key}: {value}")

Global Variables configured:
  storeNumMinSimu: 0
  storeNumMaxSimu: 0
  storeKeyDataN: 0
  dbMasterInput: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/MasterInput.db
  dbModelsDictionary: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/ModelsDictionaryArise.db
  dbCelsius: 
  dt: 0
  ori_MI: 
  parts: 1
  tempDir: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/output/temp
  package: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/src/modfilegen
  thirdyear: 0
  dailyoutput: 1
  directorypath: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/output
  pltfolder: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/cultivars/stics
  nthreads: 2


## 4. Inspect the successive rotation

Verify that the example `idsim` expands into the expected ordered seasons before running STICS.

The calendar uses `SowingYearOffset`, while `PlantOrder` remains local to each season for mixed crops.

In [4]:
with sqlite3.connect(master_input_db) as connection:
    connection.row_factory = sqlite3.Row
    record = connection.execute(
        "SELECT * FROM SimUnitList WHERE idsim = ?", (target_idsim,)
    ).fetchone()
    if record is None:
        raise ValueError(f"Missing successive example: {target_idsim}")
    seasons = fetch_rotation_seasons(connection, dict(record))

for season in seasons:
    crops = ", ".join(plant["SpeciesName"] for plant in season["Plants"])
    print(
        f"Season {season['SeasonOrder']}: {crops}; "
        f"period={season['StartDate']} -> {season['EndDate']}; "
        f"mixed={season['IsMixedCrop']}"
    )

assert [season["SeasonOrder"] for season in seasons] == [1, 2, 3]

Season 1: maize; period=2000-04-29 -> 2000-12-15; mixed=False
Season 2: peanut; period=2000-12-16 -> 2001-12-16; mixed=False
Season 3: maize; period=2001-12-17 -> 2002-12-16; mixed=False


## 5. Run the STICS v11 successive converter

In [5]:
print("Starting STICS v11 successive conversion...")
print("=" * 60)

try:
    result_path = main() if not GlobalVariables.get("skipSecondRun") else result_path
    print("\n" + "=" * 60)
    print(f"STICS successive conversion completed: {result_path}")
except Exception as e:
    print("\n" + "=" * 60)
    print(f"Error during successive conversion: {e}")
    import traceback
    traceback.print_exc()

Starting STICS v11 successive conversion...
STICS routing: 3 standard, 1 successive simulation(s)


dbMasterInput: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/MasterInput.db
dbModelsDictionary: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/ModelsDictionaryArise.db
Indexes created successfully!


📊 Total individual STICS simulations to process: 6 (3 standard + 3 successive seasons from 1 rotation(s))


SimUnitList rows selected: 4


Balanced into 2 sub-batches (estimated season loads: [3, 3])


...

✅ Results saved to /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/output/020c3c5d-dc07-4ba4-a74c-f714a6b923a5_stics.csv


✅ 9 rows inserted into SummaryOutput.


✅ 2348 rows inserted into SticsDailyOutput.


✅ 42350 rows inserted into SticsProfile.



STICS successive conversion completed: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/output/020c3c5d-dc07-4ba4-a74c-f714a6b923a5_stics.csv


In [6]:
# Keep the historical second run cell disabled: the unified main already runs all rows.
GlobalVariables["skipSecondRun"] = True



In [7]:
print("Starting STICS v11 successive conversion...")
print("=" * 60)

try:
    result_path = main() if not GlobalVariables.get("skipSecondRun") else result_path
    print("\n" + "=" * 60)
    print(f"STICS successive conversion completed: {result_path}")
except Exception as e:
    print("\n" + "=" * 60)
    print(f"Error during successive conversion: {e}")
    import traceback
    traceback.print_exc()

Starting STICS v11 successive conversion...

STICS successive conversion completed: /mnt/d/Mes Donnees/TCMP/github/ModFileGen/tests/successive/output/020c3c5d-dc07-4ba4-a74c-f714a6b923a5_stics.csv
